# Dynamic LoRA scale — episode 0, adaptive_gt, all scenarios

Reads the adaptive_gt JSON dumps in `trained_models/LoraF_invi_visi_rank_1/test/` and
plots `robot.lora_scale` (the per-step dynamic scale set by the adaptive controller)
for episode 0 of each scenario.

The GT-friendly ratio (fraction of humans with `actual_friendly == True` at that step)
is overlaid as a dashed line — it's the signal the `adaptive_gt` controller reacts to.

In [ ]:
import json
import os
import matplotlib.pyplot as plt

MODEL_DIR = 'trained_models/LoraF_invi_visi_rank_1'
TEST_DIR = os.path.join(MODEL_DIR, 'test')
SEED = 42
EPISODE = 0

SCENARIOS = {
    'all_aware':    'seperate_all_aware',
    'all_ignorant': 'seperate_all_ignorant',
    'mixed_5050':   'seperate_mixed_5050',
    'cluster':      'cluster_aware_ignorant',
}

In [ ]:
def load_episode(internal_name, seed, ep_idx):
    path = os.path.join(TEST_DIR, f'{internal_name}_adaptive_gt_exp{seed}.json')
    with open(path) as f:
        data = json.load(f)
    ep = data['episodes'][ep_idx]
    steps = [sd['step'] for sd in ep['steps_data']]
    scales = [sd['robot']['lora_scale'] for sd in ep['steps_data']]
    gt_ratios = []
    for sd in ep['steps_data']:
        af = [int(h['actual_friendly']) for h in sd['humans']]
        gt_ratios.append(sum(af) / len(af) if af else 0.0)
    return steps, scales, gt_ratios, ep['result']

results = {short: load_episode(internal, SEED, EPISODE)
           for short, internal in SCENARIOS.items()}
for short, (steps, scales, gt, res) in results.items():
    print(f'{short:<14} steps={len(steps):3d}  result={res:<10s}  '
          f'mean_scale={sum(scales)/len(scales):.3f}  mean_gt={sum(gt)/len(gt):.3f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=False, sharey=True)
axes = axes.flatten()

for ax, (short, (steps, scales, gt, res)) in zip(axes, results.items()):
    ax.plot(steps, scales, color='tab:blue', lw=2, label='dynamic LoRA scale')
    ax.plot(steps, gt, color='tab:red', lw=1.2, ls='--', alpha=0.8,
            label='GT friendly ratio')
    ax.set_title(f'{short}  (episode {EPISODE}, {res})')
    ax.set_xlabel('step')
    ax.set_ylabel('value')
    ax.set_ylim(-0.05, 1.05)
    ax.grid(alpha=0.3)
    ax.legend(loc='best', fontsize=9)

fig.suptitle(f'Dynamic LoRA scale per step — adaptive_gt, seed {SEED}',
             fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = {'all_aware': 'tab:green', 'all_ignorant': 'tab:red',
          'mixed_5050': 'tab:blue', 'cluster': 'tab:orange'}
for short, (steps, scales, _gt, res) in results.items():
    ax.plot(steps, scales, color=colors[short], lw=1.8,
            label=f'{short} ({res})')
ax.set_xlabel('step')
ax.set_ylabel('dynamic LoRA scale')
ax.set_title(f'Dynamic LoRA scale — episode {EPISODE}, adaptive_gt, seed {SEED}')
ax.set_ylim(-0.05, 1.05)
ax.grid(alpha=0.3)
ax.legend(loc='best')
fig.tight_layout()
plt.show()

---
# Dynamic LoRA scale — episode 0, adaptive_pred, all scenarios

Same plot for the predictor-driven controller (`adaptive_pred`). The PRED ratio
(dotted) is what the FriendlyPredictor said at that step (`humans[i].is_friendly`);
the GT ratio (dashed) is the actual awareness. The gap between them is predictor
error — when it's large, the dynamic scale is reacting to a wrong signal.

In [ ]:
# NOTE: as of 2026-05-28 no adaptive_pred test runs exist that used the new
# 5-seed high-accuracy predictor at trained_models/LoraF_invi_visi_rank_1/
# tune_deep/V0_baseline_h192_L3_H4_d0.1_seeds42_1000_2000_3000_4000_100ep.pth.
# All adaptive_pred JSONs in test/ are from older DAgger iterations whose
# per-(human,step) ep-0 accuracy ranges 48-59% across the 4 scenarios.
#
# Best of what's on disk:
#   iter1_seed337769  -> mean ep0 acc 0.580 (balanced across scenarios)
#   iter1_seed409623  -> mean ep0 acc 0.590 (best mean, biased toward "ignorant")
#   iter2_seed144713  -> mean ep0 acc 0.489
#   iter3_seed156586  -> mean ep0 acc 0.480
#
# To use the actually high-accuracy predictor, re-run adaptive_pred tests after
# copying the tune_deep checkpoint over the canonical friendly_predictor.pth.
PRED_EXP_ID = 'dagger_h20_adaptive_iter1_seed409623'

def load_episode_pred(internal_name, exp_id, ep_idx):
    path = os.path.join(TEST_DIR, f'{internal_name}_adaptive_pred_exp{exp_id}.json')
    with open(path) as f:
        data = json.load(f)
    ep = data['episodes'][ep_idx]
    steps = [sd['step'] for sd in ep['steps_data']]
    scales = [sd['robot']['lora_scale'] for sd in ep['steps_data']]
    gt_ratios, pred_ratios = [], []
    correct, total = 0, 0
    for sd in ep['steps_data']:
        af = [int(h['actual_friendly']) for h in sd['humans']]
        pf = [int(h['is_friendly']) for h in sd['humans']]
        gt_ratios.append(sum(af) / len(af) if af else 0.0)
        pred_ratios.append(sum(pf) / len(pf) if pf else 0.0)
        correct += sum(int(a == p) for a, p in zip(af, pf))
        total   += len(af)
    pred_acc = correct / total if total else 0.0
    return steps, scales, gt_ratios, pred_ratios, ep['result'], pred_acc

results_pred = {short: load_episode_pred(internal, PRED_EXP_ID, EPISODE)
                for short, internal in SCENARIOS.items()}
for short, (steps, scales, gt, pred, res, acc) in results_pred.items():
    print(f'{short:<14} steps={len(steps):3d}  result={res:<10s}  '
          f'mean_scale={sum(scales)/len(scales):.3f}  '
          f'mean_gt={sum(gt)/len(gt):.3f}  '
          f'mean_pred={sum(pred)/len(pred):.3f}  '
          f'pred_acc={acc:.3f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=False, sharey=True)
axes = axes.flatten()

for ax, (short, (steps, scales, gt, pred, res, acc)) in zip(axes, results_pred.items()):
    ax.plot(steps, scales, color='tab:blue', lw=2, label='dynamic LoRA scale')
    ax.plot(steps, gt, color='tab:red', lw=1.2, ls='--', alpha=0.8,
            label='GT friendly ratio')
    ax.plot(steps, pred, color='tab:purple', lw=1.2, ls=':', alpha=0.9,
            label='pred friendly ratio')
    ax.set_title(f'{short}  (episode {EPISODE}, {res}, pred acc={acc*100:.1f}%)')
    ax.set_xlabel('step')
    ax.set_ylabel('value')
    ax.set_ylim(-0.05, 1.05)
    ax.grid(alpha=0.3)
    ax.legend(loc='best', fontsize=9)

fig.suptitle(f'Dynamic LoRA scale per step — adaptive_pred, exp_id {PRED_EXP_ID}',
             fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = {'all_aware': 'tab:green', 'all_ignorant': 'tab:red',
          'mixed_5050': 'tab:blue', 'cluster': 'tab:orange'}
for short, (steps, scales, _gt, _pred, res, acc) in results_pred.items():
    ax.plot(steps, scales, color=colors[short], lw=1.8,
            label=f'{short} ({res}, acc={acc*100:.1f}%)')
ax.set_xlabel('step')
ax.set_ylabel('dynamic LoRA scale')
ax.set_title(f'Dynamic LoRA scale — episode {EPISODE}, adaptive_pred')
ax.set_ylim(-0.05, 1.05)
ax.grid(alpha=0.3)
ax.legend(loc='best')
fig.tight_layout()
plt.show()

## Side-by-side: adaptive_gt vs adaptive_pred dynamic scale

One row per scenario. Left column = GT-driven scale, right column = predictor-driven
scale. Useful for spotting where the predictor's noise causes the controller to swing
differently from the GT-perfect baseline.

In [ ]:
fig, axes = plt.subplots(len(SCENARIOS), 2, figsize=(12, 2.6 * len(SCENARIOS)),
                          sharex=False, sharey=True)

for row, short in enumerate(SCENARIOS):
    steps_gt, scales_gt, gt_gt, res_gt = results[short]
    steps_pr, scales_pr, gt_pr, pred_pr, res_pr, acc_pr = results_pred[short]

    ax = axes[row, 0]
    ax.plot(steps_gt, scales_gt, color='tab:blue', lw=2, label='scale')
    ax.plot(steps_gt, gt_gt, color='tab:red', lw=1.2, ls='--', alpha=0.8, label='GT')
    ax.set_title(f'adaptive_gt — {short} ({res_gt})')
    ax.set_ylim(-0.05, 1.05); ax.grid(alpha=0.3); ax.set_ylabel('value')
    if row == 0: ax.legend(loc='best', fontsize=8)

    ax = axes[row, 1]
    ax.plot(steps_pr, scales_pr, color='tab:blue', lw=2, label='scale')
    ax.plot(steps_pr, gt_pr, color='tab:red', lw=1.2, ls='--', alpha=0.8, label='GT')
    ax.plot(steps_pr, pred_pr, color='tab:purple', lw=1.2, ls=':', alpha=0.9, label='pred')
    ax.set_title(f'adaptive_pred — {short} ({res_pr}, pred acc={acc_pr*100:.1f}%)')
    ax.set_ylim(-0.05, 1.05); ax.grid(alpha=0.3)
    if row == 0: ax.legend(loc='best', fontsize=8)
    if row == len(SCENARIOS) - 1:
        axes[row, 0].set_xlabel('step'); axes[row, 1].set_xlabel('step')

fig.tight_layout()
plt.show()

---
## Awareness-signal accuracy — `adaptive_discrepancy` vs GT

For `adaptive_discrepancy`, the `is_friendly` field in the dump is the discrepancy
detector's classification (per-step, per-human). Threshold = 0.15, M = 1.
Below: per-(human, step) accuracy across the **full** 5-episode disc5 run, and a
side-by-side table vs the `adaptive_pred` run that used the new 5-seed predictor
(`good5seed`).

In [ ]:
DISC_EXP_ID = 'disc5'      # adaptive_discrepancy run (threshold=0.15, M=1, 5 eps)
PRED_RUN_ID = 'good5seed'  # adaptive_pred run with 5-seed canonical predictor

def acc_full(internal_name, behaviour, exp_id):
    path = os.path.join(TEST_DIR,
        f'{internal_name}_{behaviour}_exp{exp_id}.json')
    d = json.load(open(path))
    tp = tn = fp = fn = 0
    for ep in d['episodes']:
        for sd in ep['steps_data']:
            for h in sd['humans']:
                af = bool(h['actual_friendly'])
                pf = bool(h['is_friendly'])
                if   af and     pf: tp += 1
                elif (not af) and (not pf): tn += 1
                elif (not af) and pf: fp += 1
                else: fn += 1
    n = tp + tn + fp + fn
    return dict(n=n, acc=(tp + tn) / n if n else 0.0,
                tp=tp, tn=tn, fp=fp, fn=fn)

rows = []
for short, internal in SCENARIOS.items():
    d = acc_full(internal, 'adaptive_discrepancy', DISC_EXP_ID)
    p = acc_full(internal, 'adaptive_pred',        PRED_RUN_ID)
    rows.append((short, d, p))

# ---------- table ----------
hdr = (f'{"scenario":<14} | '
       f'{"disc acc":>9} {"TP":>5} {"TN":>5} {"FP":>5} {"FN":>5}'
       f'   ||   {"pred acc":>9} {"TP":>5} {"TN":>5} {"FP":>5} {"FN":>5}'
       f'   ||   {"Δ (pred-disc)":>13}')
print(hdr)
print('-' * len(hdr))
for short, d, p in rows:
    print(f'{short:<14} | '
          f'{d["acc"]:>9.3f} {d["tp"]:>5} {d["tn"]:>5} {d["fp"]:>5} {d["fn"]:>5}'
          f'   ||   '
          f'{p["acc"]:>9.3f} {p["tp"]:>5} {p["tn"]:>5} {p["fp"]:>5} {p["fn"]:>5}'
          f'   ||   {(p["acc"]-d["acc"]):>+13.3f}')

# ---------- summary line ----------
mean_d = sum(d['acc'] for _, d, _ in rows) / len(rows)
mean_p = sum(p['acc'] for _, _, p in rows) / len(rows)
print(f'\nmean acc:  discrepancy={mean_d:.3f}   pred(good5seed)={mean_p:.3f}   '
      f'delta={mean_p-mean_d:+.3f}')